# 7.2 Reasoning patterns: rules, resolution and chaining: practice notebook

Prove things without listing every model. Use inference rules, turn sentences into clauses and resolve them, chain through if then rules forwards and backwards, and see how SAT solvers like DPLL decide huge problems.

How to use it: run the cells from top to bottom (Shift + Enter). First read and run the examples. Then try each task in its starter cell and run the check cell under it. The full solutions are at the end, so try on your own first.

Everything here is plain Python 3. No extra packages are needed.

## Setup

In [ ]:
# A small helper. check(tests) runs each test of a task and tells you which ones pass.
# Each test is a pair: the call to make, and the answer we expect.
import json

def check(tests):
    passed = 0
    for call, expected in tests:
        try:
            got = eval(call)
        except Exception as error:
            got = "error: " + str(error)
        same = json.dumps(got, sort_keys=True, default=repr) == json.dumps(expected, sort_keys=True, default=repr)
        if same:
            passed = passed + 1
            print("PASS ", call)
        else:
            print("FAIL ", call, " expected", expected, " but got", got)
    if passed == len(tests):
        print("All", len(tests), "tests pass. Well done!")
    else:
        print(passed, "of", len(tests), "tests pass. Keep going, or look at the solution below.")

## Examples from the chapter

### Resolution refutation for the wumpus question

In [ ]:
# Resolution refutation for the wumpus question: does the KB entail "no pit in [1,2]"?
# A clause is a set of literals. "-P12" means not P12.
kb = [{"-B11", "P12", "P21"}, {"-P12", "B11"}, {"-P21", "B11"}, {"-B11"}]
goal_negated = [{"P12"}]                 # the opposite of what we want to prove

def opposite(lit):
    return lit[1:] if lit.startswith("-") else "-" + lit

def resolve(a, b):
    out = []
    for lit in sorted(a):                 # sorted, so the order is the same every run
        if opposite(lit) in b:
            out.append((a - {lit}) | (b - {opposite(lit)}))
    return out

clauses = kb + goal_negated
new_ones = goal_negated
while new_ones:
    found = []
    for a in new_ones:
        for b in clauses:
            for r in resolve(a, b):
                if r not in clauses and r not in found:
                    print("resolve", sorted(a), "with", sorted(b), "->", sorted(r) if r else "{} empty")
                    found.append(r)
    if set() in found:
        print("empty clause: the KB entails not P12")
        break
    clauses = clauses + found
    new_ones = found

### Forward chaining with counts and an agenda

In [ ]:
# Forward chaining on the textbook Horn knowledge base. Each rule is (premises, conclusion).
rules = [(["P"], "Q"), (["L", "M"], "P"), (["B", "L"], "M"), (["A", "P"], "L"), (["A", "B"], "L")]
facts = ["A", "B"]
query = "Q"

count = [len(p) for p, c in rules]       # premises still unknown for each rule
known = set()
agenda = list(facts)
while agenda:
    s = agenda.pop(0)
    if s == query:
        print("proved", query)
        break
    if s in known:
        continue
    known.add(s)
    for i in range(len(rules)):
        if s in rules[i][0]:
            count[i] = count[i] - 1
            if count[i] == 0:
                print("rule", i + 1, "fires:", rules[i][1])
                agenda.append(rules[i][1])

### DPLL with unit clauses

In [ ]:
# DPLL on a small CNF formula. Literals are numbers: 1 means x1, -1 means not x1.
clauses = [[1, 2], [-1, 3], [-2, 3], [-3, 4], [-4, -1]]

def simplify(cls, lit):
    out = []
    for c in cls:
        if lit in c:
            continue                       # the clause is already true
        out.append([x for x in c if x != -lit])   # drop the false literal
    return out

def dpll(cls, model):
    if not cls:
        return model                       # every clause is true
    if [] in cls:
        return None                        # some clause is false
    for c in cls:
        if len(c) == 1:                    # unit clause: no choice
            return dpll(simplify(cls, c[0]), model + [c[0]])
    lit = cls[0][0]                        # otherwise guess, and undo if it fails
    result = dpll(simplify(cls, lit), model + [lit])
    if result is None:
        result = dpll(simplify(cls, -lit), model + [-lit])
    return result

print("model:", dpll(clauses, []))

## Your turn

### Task 1: The resolution rule

A clause is a Python set of literal strings, where `"-P"` means ¬P. Write `resolve(a, b)` that returns the list of all resolvents, each as a sorted list. Resolve on one pair at a time. Leave out resolvents that contain a literal and its opposite (they are always true).

Hint: For each literal in a whose opposite is in b, join the two clauses without that pair. Skip a result that holds both X and -X.

In [ ]:
def resolve(a, b):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("resolve({'P', 'Q'}, {'-Q', 'R'})", [['P', 'R']]), ("resolve({'P12'}, {'-P12', 'B11'})", [['B11']]), ("resolve({'B11'}, {'-B11'})", [[]]), ("resolve({'P', 'Q'}, {'-P', '-Q'})", []), ("resolve({'A'}, {'B'})", [])])

### Task 2: Forward chaining

Write `fc_entails(rules, facts, q)`. rules is a list of `(premises, conclusion)` pairs, facts a list of symbols known to be true. Return True if forward chaining reaches q.

Hint: Keep a count of unknown premises per rule and an agenda of symbols to process. When a count reaches 0, add that rule's conclusion to the agenda.

In [ ]:
def fc_entails(rules, facts, q):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([("fc_entails([(['P'], 'Q'), (['L', 'M'], 'P'), (['B', 'L'], 'M'), (['A', 'P'], 'L'), (['A', 'B'], 'L')], ['A', 'B'], 'Q')", True), ("fc_entails([(['P'], 'Q'), (['L', 'M'], 'P'), (['B', 'L'], 'M'), (['A', 'P'], 'L')], ['A', 'B'], 'Q')", False), ("fc_entails([], ['A'], 'A')", True), ("fc_entails([(['A'], 'B'), (['B'], 'C'), (['C', 'D'], 'E')], ['A'], 'E')", False), ("fc_entails([(['A'], 'B'), (['B'], 'C'), (['C', 'A'], 'E')], ['A'], 'E')", True)])

### Task 3: A DPLL solver

Write `dpll_sat(clauses)`. Clauses are lists of non zero integers: 3 means x3 and −3 means ¬x3. Return True if some assignment makes every clause true, else False. Use early termination and unit clauses.

Hint: simplify(clauses, lit) removes clauses that contain lit and removes -lit from the others. Then: no clauses means True, an empty clause means False, a unit clause forces its literal, otherwise try both values.

In [ ]:
def dpll_sat(clauses):
    # your code here
    pass

Run this cell to check your answer:

In [ ]:
check([('dpll_sat([[1, 2], [-1, 3], [-2, 3], [-3, 4], [-4, -1]])', True), ('dpll_sat([[1], [-1]])', False), ('dpll_sat([[1, 2], [-1, 2], [1, -2], [-1, -2]])', False), ('dpll_sat([])', True), ('dpll_sat([[1, 2, 3], [-1], [-2], [-3, 4], [-4]])', False)])

## Solutions

Try the tasks yourself first. Running a solution cell replaces your own function with the model one.

### Solution to task 1: The resolution rule

Test 4 is the classic mistake: both possible resolvents are always true, so they are dropped and the answer is an empty list.

In [ ]:
def opposite(lit):
    if lit.startswith("-"):
        return lit[1:]
    return "-" + lit

def resolve(a, b):
    out = []
    for lit in sorted(a):
        if opposite(lit) in b:
            rest = (a - {lit}) | (b - {opposite(lit)})
            useless = False
            for x in rest:
                if opposite(x) in rest:
                    useless = True
            if not useless:
                out.append(sorted(rest))
    return out

In [ ]:
check([("resolve({'P', 'Q'}, {'-Q', 'R'})", [['P', 'R']]), ("resolve({'P12'}, {'-P12', 'B11'})", [['B11']]), ("resolve({'B11'}, {'-B11'})", [[]]), ("resolve({'P', 'Q'}, {'-P', '-Q'})", []), ("resolve({'A'}, {'B'})", [])])

### Solution to task 2: Forward chaining

Test 2 removes the rule A ∧ B ⇒ L. Now L needs P and P needs L, so nothing fires and Q is not reached.

In [ ]:
def fc_entails(rules, facts, q):
    count = []
    for premises, conclusion in rules:
        count.append(len(premises))
    known = set()
    agenda = list(facts)
    while agenda:
        s = agenda.pop(0)
        if s == q:
            return True
        if s in known:
            continue
        known.add(s)
        for i in range(len(rules)):
            if s in rules[i][0]:
                count[i] = count[i] - 1
                if count[i] == 0:
                    agenda.append(rules[i][1])
    return False

In [ ]:
check([("fc_entails([(['P'], 'Q'), (['L', 'M'], 'P'), (['B', 'L'], 'M'), (['A', 'P'], 'L'), (['A', 'B'], 'L')], ['A', 'B'], 'Q')", True), ("fc_entails([(['P'], 'Q'), (['L', 'M'], 'P'), (['B', 'L'], 'M'), (['A', 'P'], 'L')], ['A', 'B'], 'Q')", False), ("fc_entails([], ['A'], 'A')", True), ("fc_entails([(['A'], 'B'), (['B'], 'C'), (['C', 'D'], 'E')], ['A'], 'E')", False), ("fc_entails([(['A'], 'B'), (['B'], 'C'), (['C', 'A'], 'E')], ['A'], 'E')", True)])

### Solution to task 3: A DPLL solver

Test 3 lists all four clauses over x1 and x2, so every assignment breaks one. Test 5 forces x1, x2 and x3 false, which breaks the first clause.

In [ ]:
def simplify(clauses, lit):
    out = []
    for c in clauses:
        if lit in c:
            continue
        smaller = []
        for x in c:
            if x != -lit:
                smaller.append(x)
        out.append(smaller)
    return out

def dpll_sat(clauses):
    if len(clauses) == 0:
        return True
    for c in clauses:
        if len(c) == 0:
            return False
    for c in clauses:
        if len(c) == 1:
            return dpll_sat(simplify(clauses, c[0]))
    lit = clauses[0][0]
    if dpll_sat(simplify(clauses, lit)):
        return True
    return dpll_sat(simplify(clauses, -lit))

In [ ]:
check([('dpll_sat([[1, 2], [-1, 3], [-2, 3], [-3, 4], [-4, -1]])', True), ('dpll_sat([[1], [-1]])', False), ('dpll_sat([[1, 2], [-1, 2], [1, -2], [-1, -2]])', False), ('dpll_sat([])', True), ('dpll_sat([[1, 2, 3], [-1], [-2], [-3, 4], [-4]])', False)])